In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score

df = pd.read_csv('../data/job_salary_prediction_dataset.csv')
df.head()

x = df.drop('salary',axis=1)
y = df['salary']

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

num = x.select_dtypes(include=['int64','float64']).columns.tolist()
cat = x.select_dtypes(include=['object']).columns.tolist()

print(num)
print(cat)

num_pipe = Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())])
cat_pipe = Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('one',OneHotEncoder(handle_unknown='ignore',drop='first'))])

preprocess = ColumnTransformer([('num',num_pipe,num),('cat',cat_pipe,cat)])

model = Pipeline([('preprocess',preprocess),('model',XGBRegressor(learning_rate=0.1,max_depth=6,n_estimators=200))])

model.fit(x_train,y_train)
pred = model.predict(x_test)
print(r2_score(pred,y_test))

/tmp/ipykernel_14389/1468099006.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat = x.select_dtypes(include=['object']).columns.tolist()


['experience_years', 'skills_count', 'certifications']
['job_title', 'education_level', 'industry', 'company_size', 'location', 'remote_work']
0.9725756049156189


In [ ]:
import joblib
from sklearn.metrics import r2_score

model = joblib.load("model.pkl")
x_test,y_test = joblib.load('set.pkl')

prediction = model.predict(x_test)
acc = r2_score(prediction,y_test)

print('metrics',acc)

if acc <= 0.90:
    print("model was better at performing")
else:
    print("model was worst condition")